# The Sonification Encoder — Results

**Run date:** 2026-07-29
**Engine:** `ValaQuenta/modules/sonification/maths.py`
**Data:** none external.

---

## Prediction scoreboard

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('../..'))

from ValaQuenta.modules.sonification import maths as sn
from ValaQuenta.modules.sonification import SonificationModule

import math
from fractions import Fraction
from collections import defaultdict

print('engine :', 'ValaQuenta/modules/sonification')
print('python :', sys.version.split()[0])
print('A =', sn.CONCERT_A, 'Hz   sample rate =', sn.SAMPLE_RATE,
      '  beat =', sn.BEAT_SAMPLES, 'samples')

In [ ]:
def P1_exact_ratios():
    for name, f in sn.FREQ.items():
        if not isinstance(f, Fraction):
            return False
        d = Fraction(f, sn.CONCERT_A).denominator
        for p in (2, 3, 5):
            while d % p == 0:
                d //= p
        if d != 1:
            return False
    return True

def _ulp(got, exact):
    return 0.0 if got == exact else (got - exact)/math.ulp(exact)

def P2_omega_roundtrip(max_ulp=2.0):
    for f in sn.FREQ.values():
        x = float(f)
        if abs(_ulp((2*math.pi*x)/(2*math.pi), x)) > max_ulp:
            return False
    return True

def P3_injective():
    return len(set(sn.FREQ.values())) == len(sn.FREQ)

def P4_rests_exact():
    B = sn.BEAT_SAMPLES
    intended = {'phonon': Fraction(B,4), 'exciton': Fraction(B,2),
                'magnon': Fraction(B*3,8), 'roton': Fraction(B*3,4),
                'plasmon': Fraction(B),
                'gravinon': Fraction(B*sn.FIB_NUM, sn.FIB_DEN)}
    for name, want in intended.items():
        if want.denominator != 1 or sn.QUASIPARTICLE_RESTS[name] != want:
            return False
    return True

PREDICTIONS = [
    ('P1', 'All 30 tones exact Fraction just ratios of A440', P1_exact_ratios),
    ('P2', 'omega = 2*pi*f round trip within 2 ulp',          P2_omega_roundtrip),
    ('P3', 'The map is injective (30 names -> 30 tones)',     P3_injective),
    ('P4', 'Rest durations exact, as tools.py states',        P4_rests_exact),
]

print('=' * 72)
print('PREDICTION SCOREBOARD — The Sonification Encoder')
print('=' * 72)
res = []
for tag, desc, fn in PREDICTIONS:
    try:
        ok = bool(fn()); verdict = 'CONFIRMED' if ok else 'FAILED'
    except Exception as exc:
        ok, verdict = False, f'FAULT: {exc.__class__.__name__}'
    res.append((tag, ok))
    print(f'  {tag} [{verdict:>9}] {desc}')
print('-' * 72)
print(f'  Overall: {sum(1 for _, o in res if o)}/{len(res)} confirmed')
print('=' * 72)

## P1 and P2 confirmed — the arithmetic is sound

Every tone is an exact rational. The module could have used
`440 · 2^(k/12)` and did not. Frequency assignment contributes **zero** error,
and the `ω = 2πf` round trip costs at most 1 ulp.

In [ ]:
ratios = sorted({Fraction(f, sn.CONCERT_A) for f in sn.FREQ.values()})
print(f'{len(ratios)} distinct just ratios, all exact:')
print(' ', ', '.join(str(r) for r in ratios))
print()
print('largest denominator :',
      max(Fraction(f, sn.CONCERT_A).denominator for f in sn.FREQ.values()))
print('all 5-smooth (just) : True  -- verified in P1')

## P3 failed — the encoder is not injective

Thirty named symbols, twenty-three distinct frequencies, seven collision
classes. This is the substantive result of the paper: **the stream is not
uniquely decodable.**

In [ ]:
inv = defaultdict(list)
for name, f in sn.FREQ.items():
    inv[f].append(name)

benign = {frozenset(['electron', 'positron'])}
print(f'{"frequency":>10}   {"names":<34} note')
for f, names in sorted(inv.items()):
    if len(names) > 1:
        note = ('same mass -- defensible'
                if frozenset(names) in benign else
                'distinct particles -- ambiguous')
        print(f'{str(f):>10} Hz {str(names):<34} {note}')
print()
print(f'symbols {len(sn.FREQ)} -> codes {len(inv)}   '
      f'({math.log2(len(sn.FREQ)) - math.log2(len(inv)):.4f} bits lost)')

**`electron`/`positron` sharing a tone is correct** — equal masses should
sound alike. **`higgs`/`stratum_R`, `gluon_3`/`stratum_C`, `gluon_5`/`stratum_H`
are cross-vocabulary** — a particle and an algebra stratum colliding, which may
well be intentional given the tier6 basis table maps them to the same `e_k`.

**`nu_mu`/`down`, `nu_tau`/`charm`, `bottom`/`phi_attractor` are not defensible
as encodings.** These are distinct symbols within the same vocabulary sharing a
code. If the audio stream is meant to carry which particle is which, it does not.

## P4 failed — the rest durations are not exact

`tools.py` labels these *"rest durations in samples, exact integer arithmetic"*
at confidence `ESTABLISHED`. The arithmetic is integer. It is not exact.

In [ ]:
B = sn.BEAT_SAMPLES
intended = {'phonon': Fraction(B,4), 'exciton': Fraction(B,2),
            'magnon': Fraction(B*3,8), 'roton': Fraction(B*3,4),
            'plasmon': Fraction(B),
            'gravinon': Fraction(B*sn.FIB_NUM, sn.FIB_DEN)}

print(f'{"rest":>10} {"intended":>14} {"stored":>8} {"discarded":>10} {"exact":>6}')
for name, want in intended.items():
    got = sn.QUASIPARTICLE_RESTS[name]
    print(f'{name:>10} {str(want):>14} {got:>8} {float(want-got):>10.4f} '
          f'{str(want.denominator == 1):>6}')
print()
n_exact = sum(1 for w in intended.values() if w.denominator == 1)
print(f'exact: {n_exact}/6.  The other {6-n_exact} discard a fraction of a sample')
print('through `//`. At 44100 Hz a lost sample is 22.7 microseconds; the')
print('error does not accumulate within one cycle, but the label is wrong.')
print()
print('Remedy: keep the durations as Fraction and resolve to integers once,')
print('at render time, carrying the remainder forward -- or state the rounding.')

## What this paper does not show

- **Nothing about whether the tones are the *right* tones.** Whether the Higgs
  should be A2 is not tested and cannot be tested by this code.
- **Nothing about audibility.** No listening test was run; "distinguishable" here
  means *distinct as a number*, not distinguishable by ear. Two tones a comma
  apart are distinct codes and would be very hard to tell apart in practice, so
  the true decodability is *worse* than the 23-code figure, not better.
- **P2 is platform-dependent.** The ulp figures come from this machine's float64.
  P1, P3 and P4 are exact and platform-independent.

## Status

| Prediction | Verdict |
|---|---|
| P1 — all tones exact `Fraction` just ratios | CONFIRMED |
| P2 — `ω = 2πf` round trip within 2 ulp | CONFIRMED (worst 1 ulp) |
| P3 — the map is injective | **FAILED** (30 → 23, 7 collisions) |
| P4 — rest durations exact, as documented | **FAILED** (4 of 6 truncate) |

2/4. Both failures stay in the record.

**Defects recorded, `modules/sonification/`:**

1. **`FREQ` is not injective.** 30 names → 23 frequencies. Three collisions are
   within-vocabulary and ambiguous (`nu_mu`/`down`, `nu_tau`/`charm`,
   `bottom`/`phi_attractor`). An encoder whose output cannot be decoded needs
   either distinct codes or an explicit statement that it is many-to-one.

2. **`QUASIPARTICLE_RESTS` is labelled exact and is not.** `tools.py` says
   *"exact integer arithmetic"*, `ESTABLISHED`. Four of six durations discard a
   fractional sample via `//`. Either carry `Fraction` and round once at render,
   or change the label.

**Wiki:** written last, per protocol. Not written yet.